# 03 -- StyleGAN2

**Goal:** a GAN whose generator is built so that a latent code controls the image *style by style*
-- coarse layers set pose and shape, fine layers set colour and texture.

**In this notebook:** the mapping network and W space, modulated/demodulated convolutions, noise inputs,
style mixing, the truncation trick, R1 regularisation, and Hub upload. Uses a compact StyleGAN2 written
for 32-64 px (`genmodels/stylegan.py`).

*Papers:* Karras et al., *A Style-Based Generator Architecture* (2019) · *Analyzing and Improving the Image Quality of StyleGAN* (2020) · *Training GANs with Limited Data* (ADA, 2020).

## 1. Theory

### From z to w: the mapping network
A DCGAN feeds $z\sim\mathcal N(0,I)$ straight into the convolutions, so the latent space is forced to
have the shape of a Gaussian even if the factors of variation in the data do not. StyleGAN first maps
$z$ through an 8-layer MLP $f$ to an intermediate latent $w=f(z)$. $W$ has no imposed distribution, and in
practice it is much more **disentangled**: straight lines in $W$ change one thing at a time.

### Styles via modulated convolutions
Synthesis starts from a **learned constant** $4\times4$ tensor. Each conv layer $i$ gets its own style
$s_i = A_i(w)$ (a learned affine map) that scales the conv weights per input channel:
$$
w'_{ijk} = s_i\, w_{ijk}, \qquad
w''_{ijk} = \frac{w'_{ijk}}{\sqrt{\sum_{i,k} (w'_{ijk})^2+\epsilon}}
$$
(**modulation**, then **demodulation** so output activations have unit variance -- StyleGAN2's fix for the
"droplet" artefacts of StyleGAN1's AdaIN). The per-sample kernels are run as one grouped convolution.

### Noise inputs
Per-pixel Gaussian noise, scaled by a learned per-layer strength, is added after each conv. It supplies
*stochastic detail* (fur, hair placement) so the styles do not have to.

### Style mixing
During training, with probability 0.9 two latents $w_1,w_2$ are used: layers before a random cutoff get $w_1$,
the rest $w_2$. This stops adjacent layers from relying on sharing one style, and is what makes
"pose from A, colours from B" possible at test time.

### Truncation trick
At sampling, $w' = \bar w + \psi\,(w-\bar w)$ with $\bar w$ the running mean of $W$. $\psi<1$ pulls samples
toward the "average" image: higher quality, less diversity. $\psi=0$ gives the mean image itself.

### Training
Non-saturating logistic loss, **lazy R1 regularisation** (penalise $\|\nabla_x D(x)\|^2$ on reals every 16 steps),
equalised learning rate, an EMA copy of $G$ for sampling, and a **minibatch-std** feature in $D$ that lets
it detect low diversity. For small datasets (Ray_dog) we add DiffAugment; StyleGAN2-ADA's adaptive
augmentation is the paper's version of the same idea.

*Left out for size:* path-length regularisation, ADA, FIR-filtered resampling, progressive skip/residual variants.

## 0. Setup

Works in three places, no edits needed:

* **Google Colab** -- the cell below downloads the code from GitHub and installs it (~1 min).
  Choose a GPU first: *Runtime -> Change runtime type -> T4 GPU*.
* **RunPod** -- run `bash scripts/runpod/setup.sh` once, then open JupyterLab (port 8888).
* **Your own machine** -- `pip install -e ".[notebook,finetune]"` in the repo, then open the notebook.

In [ ]:
import sys, pathlib, subprocess

# Where is the code? If this notebook was opened on its own (e.g. in Colab), fetch it first.
REPO_URL = "https://github.com/yar78/vision-generative-models-survey.git"
BRANCH = "claude/vibrant-noether-ftcsah"   # change to "main" once this branch is merged

REPO = pathlib.Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
if not (REPO / "genmodels").is_dir():
    REPO = pathlib.Path("vision-generative-models-survey").resolve()
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}"], check=True)
sys.path.insert(0, str(REPO))

import torch
import yaml
from genmodels import build_trainer
from genmodels.common.config import load_config
from genmodels.common.data import get_dataset
from genmodels.common.utils import get_device, load_images, nearest_neighbours, plot_history, show_grid
from genmodels.hub import push_run, resolve_repo_id
from genmodels.stylegan import StyleGANGenerator, ModulatedConv2d, r1_penalty
print("code:", REPO)
print("device:", get_device(), f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "-- no GPU: training will be slow")

### Modulated convolution, as implemented

In [ ]:
import inspect; print(inspect.getsource(ModulatedConv2d.forward))

## 2. Choose an experiment

Every experiment is one YAML file in `configs/`, shared with the command-line script,
so a run started here and a run started on a RunPod GPU are the same experiment.

* `configs/stylegan/mnist.yaml` -- the quick default.
* `configs/ray_dog/stylegan.yaml` -- your **Ray_dog** photos (`yaryar78/Ray_dog`, 258 images) at 64x64.
* `configs/stylegan/cifar10.yaml` -- CIFAR-10, a longer run.

Edit `cfg` below to change anything (steps, batch size, model width ...).

In [ ]:
CONFIG = REPO / "configs/stylegan/mnist.yaml"        # or: REPO / "configs/ray_dog/stylegan.yaml"
cfg = load_config(CONFIG)

# Override anything here, for example:
# cfg["train"]["steps"] = 5000

print(yaml.safe_dump(cfg, sort_keys=False))

### Look at the training data (with the training-time augmentation applied)

In [ ]:
train_set = get_dataset(dict(cfg["data"], num_workers=0))
print(f"{len(train_set)} training images")
x_real = torch.stack([train_set[i][0] for i in torch.randperm(len(train_set))[:64].tolist()])
show_grid(x_real, title="training data")

## 3. Build the model

`build_trainer(cfg)` creates the mapping network, synthesis network and discriminator, the optimizers and the training loop.
Outputs go to `outputs/<name>/` (on RunPod: `/workspace/genmodels/outputs/<name>/`, on the network volume).

In [ ]:
trainer = build_trainer(cfg)
print("run dir:", trainer.run_dir)
print({k: f"{v/1e6:.2f}M params" for k, v in trainer.num_params().items()})

In [ ]:
G = trainer.G
print('style inputs (num_ws):', G.num_ws)
print(G.mapping)

## 4. Train

`trainer.fit()` runs `train.steps` steps. Every `sample_every` steps it saves a fixed-seed
sample grid to `samples/`; every `ckpt_every` steps a checkpoint. Re-running the cell
(or the script) **resumes** from the checkpoint.

MNIST config: ~20-30 min on a GPU. Ray_dog: a few hours for 50k steps. Samples come from the EMA generator, which lags the live one but is much cleaner.

Already trained this config with the script (e.g. on RunPod)? Set `TRAIN = False`
and the checkpoint in the run directory is loaded instead.

### Start TensorBoard first, then train

Run this cell **before** the training cell. The dashboard updates live while training runs
(it refreshes every 30 s; the reload button at its top right forces it):

* **Scalars** -- the loss curves.
* **Images** -- the fixed-seed sample grid, with a slider over training steps.

On RunPod with the script, TensorBoard is also served on port 6006 by `scripts/runpod/train.sh`.

Started training without it? Stop the training cell (■), run this one, then run the training
cell again: it **continues from the same step** -- the model and optimizer are still in memory.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {trainer.run_dir}/tensorboard

In [ ]:
TRAIN = True

if TRAIN:
    trainer.fit()
elif trainer.ckpt_path.exists():
    trainer.load_checkpoint()
    print(f"loaded checkpoint at step {trainer.step}")
else:
    print("No checkpoint yet in", trainer.run_dir)

### Loss curves
The same curves as TensorBoard's Scalars tab, drawn here once training has stopped.

In [ ]:
plot_history(trainer.history)

## 5. Results
### Samples (EMA generator, $\psi=1$)

In [ ]:
show_grid(trainer.sample_fixed(64), title='samples')

## 6. What the architecture gives you
### Truncation sweep
Same $z$ per column; rows use $\psi = 1, 0.7, 0.5, 0.3, 0$.

In [ ]:
G_ema = trainer.net_for_sampling("G").eval()
z = torch.randn(8, G_ema.z_dim, device=trainer.device)
with torch.no_grad():
    rows = [G_ema(z, truncation=psi).clamp(-1, 1) for psi in (1.0, 0.7, 0.5, 0.3, 0.0)]
show_grid(torch.cat(rows), nrow=8, title="truncation psi = 1, 0.7, 0.5, 0.3, 0 (top to bottom)", size=9)

### Style mixing
Row 0 = source A, column 0 = source B. Each cell takes the styles of the **first `cutoff` layers from A**
and the rest from B. Small cutoff: mostly B with A's coarse layout. Large cutoff: mostly A with B's colours.

In [ ]:
with torch.no_grad():
    za = torch.randn(6, G_ema.z_dim, device=trainer.device)
    zb = torch.randn(6, G_ema.z_dim, device=trainer.device)
    wa, wb = G_ema.map(za), G_ema.map(zb)
    cutoff = G_ema.num_ws // 2   # try 2 (coarse only) or G_ema.num_ws - 2 (all but finest)
    blank = torch.ones(1, trainer.channels, trainer.image_size, trainer.image_size, device=trainer.device)  # white corner cell
    top = torch.cat([blank, G_ema.synthesis(wa[:, None].repeat(1, G_ema.num_ws, 1))])
    rows = [top]
    for j in range(6):
        ws = wa[:, None].repeat(1, G_ema.num_ws, 1).clone()
        ws[:, cutoff:] = wb[j]
        b_img = G_ema.synthesis(wb[j : j + 1, None].repeat(1, G_ema.num_ws, 1))
        rows.append(torch.cat([b_img, G_ema.synthesis(ws)]))
show_grid(torch.cat(rows).clamp(-1, 1), nrow=7, title=f"style mixing, cutoff={cutoff} of {G_ema.num_ws}", size=9)

### Noise inputs
Same $w$, different per-pixel noise. Only fine stochastic detail should change; the standard-deviation map shows where.

In [ ]:
with torch.no_grad():
    z = torch.randn(1, G_ema.z_dim, device=trainer.device).repeat(8, 1)
    imgs = G_ema(z).clamp(-1, 1)
std = imgs.std(0, keepdim=True).mean(1, keepdim=True)
show_grid(imgs, nrow=8, title="same w, different noise", size=10)
show_grid((std / std.max() * 2 - 1).repeat(1, 3, 1, 1), nrow=1, title="where the noise acts (std across samples)", size=3)

### Memorisation check

With a small dataset (Ray_dog has 258 images) a generative model can simply copy training
images. For each sample (top row) we show its nearest training image in pixel space (bottom
row). Near-identical pairs mean memorisation; similar-but-different means the model generalises.

In [ ]:
eval_set = get_dataset(dict(cfg["data"], random_crop=False, hflip=False, num_workers=0))
train_imgs = load_images(eval_set, max_images=10_000)
samples = trainer.sample_fixed(8).cpu()
nn_imgs = nearest_neighbours(samples, train_imgs)[:, 0]
show_grid(torch.cat([samples, nn_imgs]), nrow=8, title="samples (top) vs nearest training image (bottom)", size=10)

## 7. Push to the Hugging Face Hub

`push_run` uploads the run directory as a model repo **`<your-username>/{cfg['name']}`**
(private by default):

| file | |
|---|---|
| `model.safetensors`, `config.json` | weights + constructor args (`StyleGANGenerator.from_pretrained` reads these) |
| `README.md` | generated model card: dataset, steps, hardware, git commit, metrics |
| `samples.png`, `metrics.json`, `train_config.yaml` | results and the exact config |
| `runs/` | TensorBoard logs (shown in the Hub's *Training metrics* tab) |

Login once with a **write** token from https://huggingface.co/settings/tokens
(`notebook_login()` below, `huggingface-cli login`, or the `HF_TOKEN` env var). To use another
repo name set `cfg["hub"]["repo_id"]`; to publish, `cfg["hub"]["private"] = False` before training,
or `push_run(..., private=False)`.

In [ ]:
PUSH_TO_HUB = False

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, whoami
    try:
        print("logged in as", whoami()["name"])
    except Exception:
        notebook_login()
    url = push_run(trainer.run_dir)

### Reload it from the Hub
This is how anyone (including future you) uses the model.

In [ ]:
if PUSH_TO_HUB:
    from genmodels.stylegan import StyleGANGenerator
    model = StyleGANGenerator.from_pretrained(resolve_repo_id(cfg)).to(trainer.device).eval()
    show_grid(model.generate(16, truncation=0.7), title="reloaded from the Hub")

## 8. Things to try

* Ray_dog with `method.diffaugment: null` -- how quickly does $D$ overfit 258 images?
* `model.mapping_layers: 1` -- is $W$ still disentangled (style mixing still clean)?
* Project a real Ray photo into $W$ (optimise $w$ to reconstruct it) -- then edit it with style mixing.